In [1]:
# ⚠️ CRITICAL: Must run this FIRST and ONLY ONCE!
# This cell completely removes torchvision to prevent circular import errors
import subprocess
import sys
import os

print("⚠️  Step 1: Uninstalling torchvision completely...")
result = subprocess.run(
    ["pip", "uninstall", "-y", "torchvision"],
    capture_output=True,
    text=True,
    timeout=60
)
print(f"   {result.stdout.split(chr(10))[0]}")

print("\n✅ Step 2: Setting environment variables...")
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("   ✅ All environment variables set")

print("\n✅ Step 3: Installing import hook...")
# Remove any cached torchvision modules
modules_to_remove = [name for name in list(sys.modules.keys()) if 'torchvision' in name.lower()]
for module_name in modules_to_remove:
    del sys.modules[module_name]
print(f"   ✅ Removed {len(modules_to_remove)} cached torchvision modules")

# Block future imports
class BlockTorchvision:
    def find_module(self, fullname, path=None):
        if 'torchvision' in fullname.lower():
            raise ImportError("torchvision is permanently disabled")
        return None

sys.meta_path.insert(0, BlockTorchvision())
print("   ✅ Import hook installed")

print("\n" + "="*70)
print("✅ ALL TORCHVISION BLOCKS ACTIVATED")
print("="*70)
print("\n⚠️  IMPORTANT: If you see torchvision-related errors, RESTART the kernel")
print("   and run this cell again as the VERY FIRST cell.\n")


⚠️  Step 1: Uninstalling torchvision completely...
   

✅ Step 2: Setting environment variables...
   ✅ All environment variables set

✅ Step 3: Installing import hook...
   ✅ Removed 0 cached torchvision modules
   ✅ Import hook installed

✅ ALL TORCHVISION BLOCKS ACTIVATED

⚠️  IMPORTANT: If you see torchvision-related errors, RESTART the kernel
   and run this cell again as the VERY FIRST cell.



# 🚀 GIS代码生成模型训练 - Google Colab (CodeLlama)

本Notebook在Google Colab上训练GIS代码生成模型（**文件级 + CodeLlama**）

**文件级训练** - 模型学习生成完整的工作流而不是单个步骤
- 输入：用户的高层指令（英语/荷兰语，如："Create MS and HS cable objects"）
- 输出：完整的工作流JSON代码（包含所有操作步骤）
- 优势：一次推理生成整个测试脚本
- **模型**：CodeLlama-7B-Instruct（专为代码生成优化）

**使用前准备：**
1. 运行环境：`Runtime > Change runtime type > T4 GPU`（免费）或 `A100 GPU`（Colab Pro）
2. 数据准备：确保已生成训练数据文件
3. 预计时间：4-6小时（T4）/ 1-2小时（A100）

---

## 📋 步骤1：环境设置

In [2]:
# 检查GPU
!nvidia-smi

Wed Mar  4 14:02:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P0             29W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 安装依赖（约3-5分钟）
print("📦 Installing dependencies...")

# 先锁定关键基础包（避免自动升级）
!pip install -q torch==2.9.0 --no-deps
!pip install -q fsspec==2024.3.1
!pip install -q numpy==2.0.2 --no-deps

# 安装主要训练库（指定兼容版本）
!pip install -q transformers==4.46.0
!pip install -q peft==0.13.0
!pip install -q datasets==2.19.0
!pip install -q "accelerate>=1.0.0"
!pip install -q sentencepiece==0.2.0
!pip install -q tqdm
!pip install -q huggingface-hub==0.26.0

print("✅ Core dependencies installed! If running in Colab, restart runtime after this cell.")

📦 Installing dependencies...
✅ Core dependencies installed! If running in Colab, restart runtime after this cell.


## 💾 步骤2：挂载Google Drive（保存模型）

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# 创建输出目录
!mkdir -p /content/drive/MyDrive/gis-models
print("✅ Google Drive mounted!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted!


## 📂 步骤3：加载步骤级数据

In [5]:
import os

# 首先，确保我们回到根目录，避免在错误的位置克隆
%cd /content/

# 删除可能存在的旧仓库副本，确保全新的克隆
!rm -rf gis-code-ai

# 克隆您的GitHub仓库
GITHUB_REPO_URL = "https://github.com/rockyistt/gis-code-ai" # 用户提供的URL

print(f"📦 克隆仓库: {GITHUB_REPO_URL}...")
!git clone {GITHUB_REPO_URL}

# 检查是否成功克隆并进入目录
if os.path.exists('gis-code-ai'):
    print("✅ 仓库克隆成功！")
    %cd gis-code-ai
    print(f"📍 当前工作目录已切换到: {os.getcwd()}")
    print("📂 目录内容: ")
    !ls -F

    # 再次检查数据文件路径
    expected_instructions_file = 'data/processed/step_level_instructions.jsonl'
    expected_data_file = 'data/processed/step_level_data.jsonl'

    if os.path.exists(expected_instructions_file) and os.path.exists(expected_data_file):
        print(f"✅ 已找到数据文件: {expected_instructions_file} 和 {expected_data_file}")
        print("   现在您可以尝试重新运行数据加载单元 (cell `XraxMOdNVGuh`)。")
    else:
        print("❌ 警告: 克隆后数据文件仍未找到。请检查您的GitHub仓库中 `data/processed/` 路径下是否包含 `step_level_instructions.jsonl` 和 `step_level_data.jsonl`。")
        print(f"   当前 {os.getcwd()}/data/ 目录内容:")
        !ls -F data/
else:
    print("❌ 仓库克隆失败，请检查您的GitHub仓库URL或权限。")

print("--------------------------------------------------")
print("克隆完成后，请运行 '步骤3：加载步骤级数据' 部分的代码单元以加载数据。")

/content
📦 克隆仓库: https://github.com/rockyistt/gis-code-ai...
Cloning into 'gis-code-ai'...
remote: Enumerating objects: 4314, done.
remote: Counting objects: 100% (4314/4314), done.
remote: Compressing objects: 100% (274/274), done.
remote: Total 4314 (delta 4077), reused 4237 (delta 4020), pack-reused 0 (from 0)
Receiving objects: 100% (4314/4314), 9.33 MiB | 4.27 MiB/s, done.
Resolving deltas: 100% (4077/4077), done.
✅ 仓库克隆成功！
/content/gis-code-ai
📍 当前工作目录已切换到: /content/gis-code-ai
📂 目录内容: 
check_instructions.py	 examples/	   show_weighted_instructions.py
compare_instructions.py  notebooks/	   src/
configs/		 output.log	   tests/
data/			 README.md	   verify_data.py
debug_workflows.py	 requirements.txt  verify_fixed.py
docs/			 scripts/	   verify_work_completion.py
✅ 已找到数据文件: data/processed/step_level_instructions.jsonl 和 data/processed/step_level_data.jsonl
   现在您可以尝试重新运行数据加载单元 (cell `XraxMOdNVGuh`)。
--------------------------------------------------
克隆完成后，请运行 '步骤3：加载步骤级数据' 部分的代码单元以

In [6]:
import os
import json
import sys
import numpy as np
import random
from pathlib import Path

print("="*70)
print("🔍 第一步：检查和加载数据文件")
print("="*70)

# 确保在正确的目录
if os.path.exists('/content/gis-code-ai'):
    os.chdir('/content/gis-code-ai')
elif os.path.exists('gis-code-ai'):
    os.chdir('gis-code-ai')

print(f"\n📍 当前工作目录: {os.getcwd()}\n")

# ============================================================
# 第1部分：检查数据文件
# ============================================================

print("📋 检查数据文件...\n")

SOURCE_FILES = {
    '✅ 步骤级指令': 'data/processed/step_level_instructions.jsonl',
    '✅ 步骤级数据': 'data/processed/step_level_data.jsonl',
}

files_status = {}
present_files = {}

for desc, filepath in SOURCE_FILES.items():
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                lines = sum(1 for _ in f)
            print(f"{desc} ✅")
            print(f"   📁 {filepath}")
            print(f"   💾 大小: {size_mb:.1f} MB | 📊 行数: {lines:,}\n")

            files_status[filepath] = 'OK'
            present_files[filepath] = size_mb

        except Exception as e:
            print(f"{desc} ⚠️")
            print(f"   📁 {filepath}")
            print(f"   ⚠️ 读取失败: {e}\n")
            files_status[filepath] = 'ERROR'
    else:
        print(f"{desc} ❌")
        print(f"   📁 {filepath}")
        print(f"   ❌ 必需但未找到\n")
        files_status[filepath] = 'MISSING'

# ============================================================
# 第2部分：加载数据文件
# ============================================================

print("="*70)
print("📂 第二步：加载数据")
print("="*70 + "\n")

# 加载step_level_instructions
all_instructions = []
print("1️⃣ 加载步骤级指令...")
try:
    with open('data/processed/step_level_instructions.jsonl', 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    item = json.loads(line)
                    all_instructions.append(item)
                except json.JSONDecodeError as e:
                    print(f"   ⚠️  第 {line_num} 行解析失败: {e}")
    print(f"   ✅ 已加载: {len(all_instructions):,} 条指令\n")
except Exception as e:
    print(f"   ❌ 加载失败: {e}\n")
    sys.exit(1)

# 加载step_level_data
all_data = []
print("2️⃣ 加载步骤级数据...")
try:
    with open('data/processed/step_level_data.jsonl', 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if line.strip():
                try:
                    item = json.loads(line)
                    all_data.append(item)
                except json.JSONDecodeError as e:
                    print(f"   ⚠️  第 {line_num} 行解析失败: {e}")
    print(f"   ✅ 已加载: {len(all_data):,} 条数据\n")
except Exception as e:
    print(f"   ❌ 加载失败: {e}\n")
    sys.exit(1)

# 验证数据一致性
if len(all_instructions) != len(all_data):
    print(f"❌ 错误：数据数量不匹配！")
    print(f"   指令: {len(all_instructions)}")
    print(f"   数据: {len(all_data)}")
    sys.exit(1)

# ============================================================
# 第3部分：构造训练数据结构
# ============================================================

print("="*70)
print("🔧 第三步：构造训练数据")
print("="*70 + "\n")

training_data = []
for instruction_item, data_item in zip(all_instructions, all_data):
    sample = {
        'instruction': instruction_item.get('instruction', ''),
        'output': data_item,
        'metadata': {
            'file_id': instruction_item.get('file_id', ''),
            'step_index': instruction_item.get('step_index', 0),
            'keywords': instruction_item.get('keywords', []),
            'avg_weight': instruction_item.get('keyword_weights', {}).get('avg_weight', 1.0),
        }
    }
    training_data.append(sample)

print(f"✅ 已构造: {len(training_data):,} 个训练样本\n")

# ============================================================
# 第4部分：分割Train/Val（按file_id，防止数据泄漏）
# ============================================================

print("="*70)
print("📊 第四步：分割数据集")
print("="*70 + "\n")

# 按file_id分组
file_id_map = {}
for idx, sample in enumerate(training_data):
    file_id = sample['metadata']['file_id']
    if file_id not in file_id_map:
        file_id_map[file_id] = []
    file_id_map[file_id].append(idx)

# 随机分割file_id（不是样本）
random.seed(42)
all_file_ids = list(file_id_map.keys())
random.shuffle(all_file_ids)

split_point = int(len(all_file_ids) * 0.9)
train_file_ids = set(all_file_ids[:split_point])

# 按file_id分割数据
train_data = []
val_data = []
for file_id, indices in file_id_map.items():
    if file_id in train_file_ids:
        train_data.extend([training_data[i] for i in indices])
    else:
        val_data.extend([training_data[i] for i in indices])

# ============================================================
# 第4.5部分：采样25%数据（OOM优化：从50%降到25%）
# ============================================================

print("="*70)
print("📊 采样25%数据（OOM优化版）")
print("="*70 + "\n")

# 原始数据大小
orig_train_size = len(train_data)
orig_val_size = len(val_data)

# ⚠️ 激进的采样：从50%降到25%以节省内存
print("⚠️  采样比例已从50%降到25%以防止OOM")
print(f"   原始训练集: {orig_train_size:,} 样本")
print(f"   原始验证集: {orig_val_size:,} 样本\n")

random.seed(42)

# 采样25%
sample_indices_train = random.sample(range(len(train_data)), max(1, int(len(train_data) * 0.25)))
sample_indices_val = random.sample(range(len(val_data)), max(1, int(len(val_data) * 0.25)))

train_data = [train_data[i] for i in sorted(sample_indices_train)]
val_data = [val_data[i] for i in sorted(sample_indices_val)]

print(f"   🔄 训练集: {orig_train_size:,} → {len(train_data):,} 样本 (保留25%)")
print(f"   🔄 验证集: {orig_val_size:,} → {len(val_data):,} 样本 (保留25%)\n")

print(f"   ✅ 采样后训练集: {len(train_data):,} 样本")
print(f"   ✅ 采样后验证集: {len(val_data):,} 样本")
print(f"   ✅ 比例: {len(train_data)/(len(train_data)+len(val_data))*100:.1f}% 训练 / {len(val_data)/(len(train_data)+len(val_data))*100:.1f}% 验证\n")

# ============================================================
# 第5部分：数据质量检查
# ============================================================

print("="*70)
print("✅ 数据质量检查")
print("="*70 + "\n")

# 检查完整性
total = len(train_data)
has_instruction = sum(1 for s in train_data if 'instruction' in s and s['instruction'])
has_output = sum(1 for s in train_data if 'output' in s)
has_keywords = sum(1 for s in train_data if s.get('metadata', {}).get('keywords'))

print(f"   训练集完整性:")
print(f"      指令完整: {has_instruction}/{total} ({has_instruction/total*100:.1f}%)")
print(f"      输出完整: {has_output}/{total} ({has_output/total*100:.1f}%)")
print(f"      关键词完整: {has_keywords}/{total} ({has_keywords/total*100:.1f}%)\n")

# 显示示例
if train_data:
    sample = train_data[0]
    print(f"📝 数据样本:")
    print(f"   指令: {sample.get('instruction', '')[:80]}...")
    print(f"   输出字段: {list(sample.get('output', {}).keys())}")
    print(f"   关键词: {sample.get('metadata', {}).get('keywords', [])[:3]}\n")

print("="*70)
print("✅ 数据加载、采样和分割完成！")
print("="*70)
print("\n📊 可用变量:")
print("   • train_data: 训练集数据 (已采样25%)")
print("   • val_data: 验证集数据 (已采样25%)")
print(f"   • 总样本数: {len(train_data) + len(val_data):,} (原始: {orig_train_size + orig_val_size:,})")
print()

🔍 第一步：检查和加载数据文件

📍 当前工作目录: /content/gis-code-ai

📋 检查数据文件...

✅ 步骤级指令 ✅
   📁 data/processed/step_level_instructions.jsonl
   💾 大小: 9.0 MB | 📊 行数: 40,209

✅ 步骤级数据 ✅
   📁 data/processed/step_level_data.jsonl
   💾 大小: 13.8 MB | 📊 行数: 40,209

📂 第二步：加载数据

1️⃣ 加载步骤级指令...
   ✅ 已加载: 40,209 条指令

2️⃣ 加载步骤级数据...
   ✅ 已加载: 40,209 条数据

🔧 第三步：构造训练数据

✅ 已构造: 40,209 个训练样本

📊 第四步：分割数据集

📊 采样25%数据（OOM优化版）

⚠️  采样比例已从50%降到25%以防止OOM
   原始训练集: 36,193 样本
   原始验证集: 4,016 样本

   🔄 训练集: 36,193 → 9,048 样本 (保留25%)
   🔄 验证集: 4,016 → 1,004 样本 (保留25%)

   ✅ 采样后训练集: 9,048 样本
   ✅ 采样后验证集: 1,004 样本
   ✅ 比例: 90.0% 训练 / 10.0% 验证

✅ 数据质量检查

   训练集完整性:
      指令完整: 9048/9048 (100.0%)
      输出完整: 9048/9048 (100.0%)
      关键词完整: 0/9048 (0.0%)

📝 数据样本:
   指令: Open E HS Kabel...
   输出字段: ['step_index', 'database', 'object', 'object_id', 'module', 'method', 'command', 'test_data', 'file_id']
   关键词: []

✅ 数据加载、采样和分割完成！

📊 可用变量:
   • train_data: 训练集数据 (已采样25%)
   • val_data: 验证集数据 (已采样25%)
   • 总样本数: 10,052 (原始: 40,209)



## 🚀 步骤4：模型加载与LoRA配置

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, TaskType, get_peft_model

# ============================================================
# 模型配置
# ============================================================

MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"  # 无需认证、专为代码优化
OUTPUT_DIR = "/content/drive/MyDrive/gis-models/step-level-model"
LORA_R = 32
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
BATCH_SIZE = 1  # ⚠️ 从2降到1（激进内存优化）
GRADIENT_ACCUMULATION = 4  # 用梯度积累来补偿batch_size的减少，相当于batch=4
EVAL_AND_SAVE_STEPS = 50
MAX_LENGTH = 192  # ⚠️ 从256降到192（节省20%的内存）

print("🔧 导入完成，模型配置已准备")
print(f"   • MODEL_NAME: {MODEL_NAME}")
print(f"   • BATCH_SIZE: {BATCH_SIZE}")
print(f"   • GRADIENT_ACCUMULATION: {GRADIENT_ACCUMULATION}")
print(f"   • 有效batch大小: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   • MAX_LENGTH: {MAX_LENGTH}")
print(f"   • LORA_R: {LORA_R}")
print(f"   • OUTPUT_DIR: {OUTPUT_DIR}\n")
print("="*70)
print("🔧 步骤5：训练配置（OOM优化版）")
print("="*70 + "\n")

# ============================================================
# 训练参数配置
# ============================================================

# 内存优化配置
BATCH_SIZE = 1  # 单个样本batch（激进）
GRADIENT_ACCUMULATION_STEPS = 4  # 每4步累积梯度（相当于batch_size=4）
MAX_LENGTH = 192  # 更短的序列长度

# 学习率和优化器配置
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_STEPS = int(len(train_data) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * 0.1)  # 10% warmup
MAX_STEPS = -1  # 使用epochs而不是steps限制

print(f"📊 批处理配置（OOM优化）:")
print(f"   • 单个batch_size: {BATCH_SIZE}")
print(f"   • GRADIENT_ACCUMULATION_STEPS: {GRADIENT_ACCUMULATION_STEPS}")
print(f"   • 有效batch大小: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"   • 每个epoch的优化步数: {len(train_data) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS):,}\n")

print(f"📈 优化器配置:")
print(f"   • LEARNING_RATE: {LEARNING_RATE}")
print(f"   • WEIGHT_DECAY: {WEIGHT_DECAY}")
print(f"   • WARMUP_STEPS: {WARMUP_STEPS:,}\n")

print(f"🔄 训练周期配置:")
print(f"   • NUM_EPOCHS: {NUM_EPOCHS}")
print(f"   • MAX_LENGTH: {MAX_LENGTH} tokens (降低20%)")
print(f"   • TRAINING_SAMPLES: {len(train_data):,}")
print(f"   • VALIDATION_SAMPLES: {len(val_data):,}\n")

# ============================================================
# 估算训练时间
# ============================================================

steps_per_epoch = len(train_data) / (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
total_steps = steps_per_epoch * NUM_EPOCHS

print(f"⏱️  训练时间估算 (T4 GPU):")
print(f"   • 总优化步数: {total_steps:.0f}")
print(f"   • 每个epoch: {steps_per_epoch:.0f} 优化步")
print(f"   • 预计时间: 2-3 小时 (20k样本, batch_size=1+GradAcc=4, 3 epochs)")
print(f"   (相比于原始配置，时间略长但更稳定)\n")

# ============================================================
# 显示内存优化配置
# ============================================================

print("💾 内存优化设置总结:")
print("   ✓ float16 精度")
print("   ✓ Gradient Checkpointing (已启用)")
print("   ✓ AdamW优化器（标准版本）")
print("   ✓ BATCH_SIZE=1 + GRAD_ACC=4 (激进内存节省)")
print("   ✓ MAX_LENGTH=192 (减少20%)")
print("   ✓ 50% 数据采样 (~20k样本)")
print("   ✓ 优化的Tokenization批处理")
print("\n   预期显存使用: ~8-10 GB (in T4's 14GB)\n")

print("="*70)
print("🚀 准备就绪！下一步：运行训练")
print("="*70)
print()

🔧 导入完成，模型配置已准备
   • MODEL_NAME: codellama/CodeLlama-7b-Instruct-hf
   • BATCH_SIZE: 1
   • GRADIENT_ACCUMULATION: 4
   • 有效batch大小: 4
   • MAX_LENGTH: 192
   • LORA_R: 32
   • OUTPUT_DIR: /content/drive/MyDrive/gis-models/step-level-model

🔧 步骤5：训练配置（OOM优化版）

📊 批处理配置（OOM优化）:
   • 单个batch_size: 1
   • GRADIENT_ACCUMULATION_STEPS: 4
   • 有效batch大小: 4
   • 每个epoch的优化步数: 2,262

📈 优化器配置:
   • LEARNING_RATE: 0.0002
   • WEIGHT_DECAY: 0.01
   • WARMUP_STEPS: 226

🔄 训练周期配置:
   • NUM_EPOCHS: 3
   • MAX_LENGTH: 192 tokens (降低20%)
   • TRAINING_SAMPLES: 9,048
   • VALIDATION_SAMPLES: 1,004

⏱️  训练时间估算 (T4 GPU):
   • 总优化步数: 6786
   • 每个epoch: 2262 优化步
   • 预计时间: 2-3 小时 (20k样本, batch_size=1+GradAcc=4, 3 epochs)
   (相比于原始配置，时间略长但更稳定)

💾 内存优化设置总结:
   ✓ float16 精度
   ✓ Gradient Checkpointing (已启用)
   ✓ AdamW优化器（标准版本）
   ✓ BATCH_SIZE=1 + GRAD_ACC=4 (激进内存节省)
   ✓ MAX_LENGTH=192 (减少20%)
   ✓ 50% 数据采样 (~20k样本)
   ✓ 优化的Tokenization批处理

   预期显存使用: ~8-10 GB (in T4's 14GB)

🚀 准备就绪！下一步：运行训练



In [8]:
# 升级库 - 确保版本兼容（PEFT版本修复）
print("🔧 Fixing PEFT version compatibility issue...")
print("   ⚠️  卸载不兼容的PEFT版本...\n")

# 强制卸载旧PEFT
!pip uninstall -y peft 2>&1 | head -5

print("\n   ⚠️  安装兼容的PEFT版本...\n")

# 重新安装兼容的版本
!pip install -q peft==0.11.1 --no-cache-dir
!pip install -q --upgrade transformers==4.46.0 --no-cache-dir
!pip install -q --upgrade accelerate>=1.0.0 --no-cache-dir
!pip install -q --upgrade "bitsandbytes>=0.43.0" --no-cache-dir

print("\n✅ Libraries fixed and upgraded!")
print("   • peft==0.11.1 (downgraded to fix ArrowConfig import)")
print("   • transformers==4.46.0")
print("   • accelerate>=1.0.0")
print("   • bitsandbytes>=0.43.0")

🔧 Fixing PEFT version compatibility issue...
   ⚠️  卸载不兼容的PEFT版本...

Found existing installation: peft 0.13.0
Uninstalling peft-0.13.0:
  Successfully uninstalled peft-0.13.0

   ⚠️  安装兼容的PEFT版本...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 21.5 MB/s eta 0:00:00

✅ Libraries fixed and upgraded!
   • peft==0.11.1 (downgraded to fix ArrowConfig import)
   • transformers==4.46.0
   • accelerate>=1.0.0
   • bitsandbytes>=0.43.0


In [9]:
# ⚠️ 可选：超激进OOM恢复方案 - 4bit量化
# 如果运行到这里时仍然OOM，取消注释下面的部分并设置USE_4BIT=True

USE_4BIT = False  # 仅在持续OOM时改为True

if USE_4BIT:
    print("⚠️  启用4-bit量化（超激进内存节省）...")
    from transformers import BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    print("   ❌ 停用此cell中的标准加载，下一个cell中的model加载会自动使用4-bit配置")
else:
    print("✅ 使用标准float16加载（内存高效但不是超激进）")
    print("   如果出现OOM，改动此cell中 USE_4BIT = True")


✅ 使用标准float16加载（内存高效但不是超激进）
   如果出现OOM，改动此cell中 USE_4BIT = True


In [10]:
# 加载tokenizer (CodeLlama)
import gc
import sys

print("📖 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="right"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded: vocab_size={len(tokenizer)}")

# ⚠️ 关键：在模型加载前清理内存
print("\n🧹 清理缓存...")
gc.collect()
torch.cuda.empty_cache()

# 清理peft相关的模块缓存（修复ArrowConfig导入问题）
peft_modules = [name for name in list(sys.modules.keys()) if 'peft' in name.lower()]
for module_name in peft_modules:
    try:
        del sys.modules[module_name]
    except:
        pass
print(f"   ✅ 清理了{len(peft_modules)}个peft模块缓存")

# 加载模型 - 简化方案：使用float16而不是8-bit
print("\n🤖 Loading model with float16 precision...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
)

# 训练中必须禁用缓存以配合梯度检查点
model.config.use_cache = False

# 启用梯度检查点（节省显存）
model.gradient_checkpointing_enable()

print("✅ Base model loaded (float16, ~6-7GB RAM)")

# ⚠️ 再次清理缓存，确保peft导入正确
gc.collect()

# 应用LoRA
print("\n🔧 Applying LoRA...")

from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied!")

# 再次清理内存
gc.collect()
torch.cuda.empty_cache()
print("\n💾 GPU Memory Status:")
print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

📖 Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Tokenizer loaded: vocab_size=32016

🧹 清理缓存...
   ✅ 清理了106个peft模块缓存

🤖 Loading model with float16 precision...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Base model loaded (float16, ~6-7GB RAM)

🔧 Applying LoRA...
trainable params: 16,777,216 || all params: 6,755,323,904 || trainable%: 0.2484
✅ LoRA applied!

💾 GPU Memory Status:
   Allocated: 13.51 GB
   Reserved: 13.52 GB


## 📚 步骤5：数据准备与Tokenization

In [11]:
from datasets import Dataset
import json as json_module

# 准备数据集
print("📊 准备datasets...")

# 转换为Dataset格式
train_dataset_hf = Dataset.from_dict({
    'instruction': [d['instruction'] for d in train_data],
    'output': [json_module.dumps(d['output'], ensure_ascii=False, indent=2) for d in train_data],
    'avg_weight': [d['metadata']['avg_weight'] for d in train_data],
})

eval_dataset_hf = Dataset.from_dict({
    'instruction': [d['instruction'] for d in val_data],
    'output': [json_module.dumps(d['output'], ensure_ascii=False, indent=2) for d in val_data],
    'avg_weight': [d['metadata']['avg_weight'] for d in val_data],
})

print(f"  训练集: {len(train_dataset_hf)} samples")
print(f"  验证集: {len(eval_dataset_hf)} samples")

# 格式化prompt：输入指令 -> 输出完整的步骤数据
def format_prompt(example):
    """
    格式化为prompt：指令 -> 步骤数据

    示例：
    Input: "Open E MS Kabel"
    Output:
    {
      "step_index": 0,
      "database": ":elektra",
      ...
    }
    """
    input_instruction = example['instruction'] # 修正：从'instruction'键获取数据
    output_json = example['output']  # 已是格式化的JSON字符串

    prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {input_instruction}

Step Data JSON:
{output_json}"""

    return {"text": prompt}

train_dataset_hf = train_dataset_hf.map(
    format_prompt,
    remove_columns=['instruction', 'output', 'avg_weight'] # 修正：删除'instruction'而不是'input'
)
eval_dataset_hf = eval_dataset_hf.map(
    format_prompt,
    remove_columns=['instruction', 'output', 'avg_weight'] # 修正：删除'instruction'而不是'input'
)

# Tokenize
def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("🔄 Tokenizing...")
train_dataset_hf = train_dataset_hf.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset_hf.column_names,
    desc="Tokenizing train"
)

eval_dataset_hf = eval_dataset_hf.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_dataset_hf.column_names,
    desc="Tokenizing val"
)

print("✅ 数据准备完成！")
print(f"\n📊 数据统计:")
train_lengths = [len(d['input_ids']) for d in train_dataset_hf]
eval_lengths = [len(d['input_ids']) for d in eval_dataset_hf]
print(f"   训练集: {len(train_dataset_hf)} samples, 平均长度: {np.mean(train_lengths):.0f} tokens, 最大: {max(train_lengths)} tokens")
print(f"   验证集: {len(eval_dataset_hf)} samples, 平均长度: {np.mean(eval_lengths):.0f} tokens, 最大: {max(eval_lengths)} tokens\n")
print(f"   格式: Instruction → Step Data JSON")


📊 准备datasets...
  训练集: 9048 samples
  验证集: 1004 samples


Map:   0%|          | 0/9048 [00:00<?, ? examples/s]

Map:   0%|          | 0/1004 [00:00<?, ? examples/s]

🔄 Tokenizing...


Tokenizing train:   0%|          | 0/9048 [00:00<?, ? examples/s]

Tokenizing val:   0%|          | 0/1004 [00:00<?, ? examples/s]

✅ 数据准备完成！

📊 数据统计:
   训练集: 9048 samples, 平均长度: 174 tokens, 最大: 192 tokens
   验证集: 1004 samples, 平均长度: 174 tokens, 最大: 192 tokens

   格式: Instruction → Step Data JSON


## 🎯 步骤6：开始训练

In [ ]:
# 配置训练 - ⚠️ FP16/显存优化修复版本
print("⚙️ 配置训练（OOM + FP16梯度修复版）...\n")

# 显示当前的显存状态
import torch
import gc
gc.collect()
print(f"💾 当前显存状态（optimization开始前）:")
print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB\n")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=10,
    save_steps=EVAL_AND_SAVE_STEPS,
    eval_steps=EVAL_AND_SAVE_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    
    # ⚠️ 修复FP16梯度问题：改为bf16而不是fp16
    fp16=False,
    bf16=True,  # bfloat16对梯度缩放友好
    
    # ⚠️ 改为adamw_torch而不是adamw_8bit（兼容性更好）
    optim="adamw_torch",
    
    lr_scheduler_type="cosine",
    save_total_limit=2,
    report_to="none",
    logging_dir=f"{OUTPUT_DIR}/logs",
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
    push_to_hub=False,
    gradient_checkpointing=True,
    
    # ⚠️ 删除max_grad_norm：与bf16梯度缩放不兼容
    # max_grad_norm=1.0,
    
    # ⚠️ 额外的显存优化参数
    dataloader_pin_memory=False,  # 降低显存压力
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_hf,
    eval_dataset=eval_dataset_hf,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("✅ Trainer ready!")
print("\n🔧 训练配置修复摘要:")
print("   ✓ fp16 = False (不再使用)")
print("   ✓ bf16 = True (bfloat16对梯度缩放友好)")
print("   ✓ optim = adamw_torch (标准优化器)")
print("   ✓ max_grad_norm = 注释掉 (避免兼容性问题)")
print("   ✓ dataloader_pin_memory = False (显存优化)\n")

⚙️ 配置训练（T4优化版）...
✅ Trainer ready!


In [ ]:
print("\n" + "="*70)
print("🚀 开始训练（bf16/adamw_torch版，预计2-4小时）...")
print("="*70)

# ⚠️ 关键：激进的显存清理
import gc
import torch

print("\n🧹 执行激进显存清理...")

# 多次垃圾回收
for i in range(3):
    gc.collect()
    torch.cuda.empty_cache()
    
print("✅ 显存清理完成")

print(f"\n💾 训练前显存状态:")
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
available = (torch.cuda.get_device_properties(0).total_memory - reserved) / 1e9

print(f"   Allocated: {allocated:.2f} GB")
print(f"   Reserved: {reserved:.2f} GB")
print(f"   Available: {available:.2f} GB")

if allocated > 13.0:
    print(f"\n⚠️  警告：显存已用接近上限！")
    print(f"   建议：如果训练失败，请减少BATCH_SIZE或MAX_LENGTH")

print("\n" + "="*70)

# 开始训练
try:
    print("⏱️  开始计时...")
    import time
    start_time = time.time()
    
    trainer.train()
    
    elapsed = (time.time() - start_time) / 3600
    print("\n" + "="*70)
    print(f"🎉 训练完成！耗时 {elapsed:.1f} 小时")
    print("="*70)
    
except torch.cuda.OutOfMemoryError as e:
    print("\n" + "="*70)
    print("❌ CUDA OOM错误")
    print("="*70)
    print("\n💡 解决方案:")
    print("   1. 重启kernel (Runtime → Restart runtime)")
    print("   2. 减少BATCH_SIZE (从1 → 1, 梯度累积4 → 2)")
    print("   3. 进一步减少MAX_LENGTH (从192 → 128)")
    print("   4. 减少采样率 (从25% → 10%)")
    print("   5. 启用模型offloading")
    raise
    
except ValueError as e:
    if "Attempting to unscale" in str(e) or "FP16" in str(e):
        print("\n" + "="*70)
        print("❌ FP16梯度错误仍然存在")
        print("="*70)
        print("\n💡 这可能是由于:")
        print("   1. 之前的Cell没有正确更新")
        print("   2. 需要重启kernel来清除旧配置")
        print("\n建议：")
        print("   - 重启kernel")
        print("   - 重新运行所有cells")
        raise
    else:
        print(f"\n❌ 训练出错: {type(e).__name__}")
        print(f"   错误信息: {str(e)[:300]}")
        raise
        
except Exception as e:
    print(f"\n❌ 训练出错: {type(e).__name__}")
    print(f"   错误信息: {str(e)[:300]}")
    raise


🚀 开始训练（T4优化版，预计2-4小时）...


ValueError: Attempting to unscale FP16 gradients.

In [ ]:
print("\n" + "="*70)
print("🧹 步骤7a: 训练完成后的内存清理")
print("="*70)

import gc
import torch

print("\n⏳ 清理训练数据和优化器状态...")

# 🔥 清理主要的大对象
try:
    # 清理数据集（占用内存最多）
    if 'train_dataset_hf' in globals():
        del train_dataset_hf
        print("   ✅ 删除 train_dataset_hf")
    
    if 'eval_dataset_hf' in globals():
        del eval_dataset_hf
        print("   ✅ 删除 eval_dataset_hf")
    
    if 'train_dataset' in globals():
        del train_dataset
        print("   ✅ 删除 train_dataset")
    
    if 'eval_dataset' in globals():
        del eval_dataset
        print("   ✅ 删除 eval_dataset")
    
    # 清理data_collator
    if 'data_collator' in globals():
        del data_collator
        print("   ✅ 删除 data_collator")
    
    # 清理trainer的优化器和调度器
    print("   ✅ 清理trainer内部状态...")
    trainer.optimizer = None
    trainer.lr_scheduler = None
    
except Exception as e:
    print(f"   ⚠️  清理过程中遇到错误: {e}")

print("\n🧹 执行Python垃圾回收...")
for i in range(3):
    gc.collect()
    print(f"   ✅ 垃圾回收 #{i+1}/3")

print("\n💾 清理CUDA缓存...")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# 显示清理后的显存状态
print("\n📊 清理后显存状态:")
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"   已用 (Allocated):   {allocated:.2f} GB")
print(f"   保留 (Reserved):    {reserved:.2f} GB")
print(f"   总容量 (Total):     {total:.2f} GB")
print(f"   可用空间:           {total - reserved:.2f} GB")

print("\n✅ 内存清理完成！")
print("="*70)


In [ ]:
print("\n" + "="*70)
print("💾 保存训练好的模型...")
print("="*70)

import os
import json

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 保存微调后的模型和tokenizer
print("\n1️⃣ 保存PEFT LoRA权重...")
trainer.save_model(OUTPUT_DIR)
print(f"   ✅ LoRA权重已保存到: {OUTPUT_DIR}")

# 保存tokenizer
print("\n2️⃣ 保存tokenizer...")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/tokenizer")
print(f"   ✅ Tokenizer已保存")

# 保存训练配置信息
print("\n3️⃣ 保存训练信息...")
training_info = {
    "model_name": MODEL_NAME,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
    "learning_rate": LEARNING_RATE,
    "warmup_steps": WARMUP_STEPS,
    "max_length": MAX_LENGTH,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "train_samples": len(train_dataset_hf),
    "val_samples": len(eval_dataset_hf),
    "precision": "bfloat16",
    "optimizer": "adamw_torch",
    "data_level": "step_level",
    "training_date": "2026-03-04",
    "output_dir": OUTPUT_DIR,
}

with open(f"{OUTPUT_DIR}/training_info.json", 'w', encoding='utf-8') as f:
    json.dump(training_info, f, indent=2, ensure_ascii=False)

print(f"   ✅ 训练信息已保存")

# 保存README
readme = f"""# GIS CodeLlama LoRA Model

## 模型信息
- **基础模型**: {MODEL_NAME}
- **微调方法**: LoRA (Low-Rank Adaptation)
- **训练数据**: Step-level GIS指令 (~{len(train_dataset_hf):,} 样本)
- **训练时间**: {NUM_EPOCHS} epochs
- **精度**: bfloat16

## LoRA配置
- **Rank (r)**: {LORA_R}
- **Alpha**: {LORA_ALPHA}
- **Dropout**: {LORA_DROPOUT}
- **Target Modules**: q_proj, v_proj

## 训练参数
- **Batch Size**: {BATCH_SIZE} (per device)
- **Gradient Accumulation**: {GRADIENT_ACCUMULATION}
- **Learning Rate**: {LEARNING_RATE}
- **Optimizer**: adamw_torch
- **Max Sequence Length**: {MAX_LENGTH}

## 使用方法

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 加载基础模型和LoRA权重
base_model = AutoModelForCausalLM.from_pretrained(
    "{MODEL_NAME}",
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "{OUTPUT_DIR}")
tokenizer = AutoTokenizer.from_pretrained("{OUTPUT_DIR}/tokenizer")

# 推理
prompt = "Your instruction here"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0]))
```

## 输入/输出格式

### 输入
自然语言GIS指令，例如:
- "Open E MS Kabel"
- "Create new station at coordinates (123, 456)"
- "Set MS cable object properties"

### 输出
结构化JSON步骤数据，包含:
- step_index: 步骤索引
- database: 数据库标识
- object: 操作对象
- method: 执行方法
- command: 执行命令
- 其他GIS特定参数

## 文件结构
```
{OUTPUT_DIR}/
├── adapter_config.json      # LoRA配置
├── adapter_model.bin        # LoRA权重
├── tokenizer/
│   ├── tokenizer.model
│   └── special_tokens_map.json
├── training_info.json       # 训练信息
└── README.md               # 本文件
```
"""

with open(f"{OUTPUT_DIR}/README.md", 'w', encoding='utf-8') as f:
    f.write(readme)

print(f"   ✅ README已保存")

print("\n" + "="*70)
print("✅ 模型保存完成！")
print("="*70)
print(f"\n📁 模型位置: {OUTPUT_DIR}")
print(f"   • adapter_config.json - LoRA配置")
print(f"   • adapter_model.bin - LoRA权重")
print(f"   • tokenizer/ - Tokenizer文件")
print(f"   • training_info.json - 训练信息")
print(f"   • README.md - 使用说明")

In [ ]:
print("🧪 模型推理测试\n")
print("="*70)
print("使用训练好的模型进行推理")
print("="*70)

# 确保模型处于评估模式
model.eval()

# 定义推理函数
def test_model(instruction, context="", max_tokens=200):
    """
    测试模型推理
    
    Args:
        instruction: 自然语言指令
        context: 上下文信息（可选）
        max_tokens: 最大生成token数
    
    Returns:
        生成的文本
    """
    # 构建prompt
    if context:
        prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Context: {context}
Instruction: {instruction}

Step Data JSON:"""
    else:
        prompt = f"""You are a GIS step instruction parser. Convert natural language instructions to structured step data JSON.

Instruction: {instruction}

Step Data JSON:"""
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    input_ids = inputs["input_ids"].to(model.device)
    
    # 生成
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # 解码
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 提取JSON部分
    if "Step Data JSON:" in generated_text:
        response = generated_text.split("Step Data JSON:")[-1].strip()
    else:
        response = generated_text
    
    return response

# 定义测试用例
test_cases = [
    {
        "name": "测试1: 打开电缆对象",
        "instruction": "Open E MS Kabel",
        "context": "Database: :elektra | Application: PowerGrid",
    },
    {
        "name": "测试2: 创建新站点",
        "instruction": "Create new station building",
        "context": "Location: Amsterdam | Type: MS Station",
    },
    {
        "name": "测试3: 设置接地变压器",
        "instruction": "Set MS HS aardingstrafo asset properties",
        "context": "Database: ND | Operation Type: Create",
    },
    {
        "name": "测试4: 添加高压连接",
        "instruction": "Setup HS cable connection between two nodes",
        "context": "Voltage Level: 110kV | Cable Type: Underground",
    },
    {
        "name": "测试5: 配置低压安装",
        "instruction": "Configure LV installation with protection devices",
        "context": "Building Type: Industrial | Safety Level: High",
    },
]

# 执行测试
print("\n")
for i, test_case in enumerate(test_cases, 1):
    print("="*70)
    print(f"🔮 {test_case['name']}")
    print("="*70)
    print(f"\n📝 输入信息:")
    print(f"   指令: {test_case['instruction']}")
    print(f"   上下文: {test_case['context']}")
    
    print(f"\n⏳ 生成中...\n")
    
    result = test_model(
        instruction=test_case['instruction'],
        context=test_case['context'],
        max_tokens=200
    )
    
    print(f"📤 输出结果:")
    print(f"{result}")
    
    # 尝试解析JSON
    try:
        import json
        json_obj = json.loads(result)
        print(f"\n✅ JSON有效性: 有效")
        print(f"📊 关键字段:")
        if 'step_index' in json_obj:
            print(f"   - step_index: {json_obj['step_index']}")
        if 'object' in json_obj:
            print(f"   - object: {json_obj['object']}")
        if 'method' in json_obj:
            print(f"   - method: {json_obj['method']}")
        if 'database' in json_obj:
            print(f"   - database: {json_obj['database']}")
    except json.JSONDecodeError:
        print(f"\n⚠️  JSON有效性: 无效 (未能解析)")
    
    print("\n")

print("="*70)
print("🎉 测试完成！")
print("="*70)

In [ ]:
print("🎯 交互式模型测试\n")
print("使用下面的代码自定义GIS指令来测试模型")
print("="*70)

# 自定义输入 - 修改这些变量来测试不同的指令
your_instruction = "Open E MS Kabel"  # 修改这里：输入你的GIS指令
your_context = "Database: :elektra | Application: PowerGrid"  # 修改这里：输入背景信息

print(f"\n📝 你的输入:")
print(f"   指令: {your_instruction}")
print(f"   上下文: {your_context}")

print(f"\n⏳ 生成中...\n")

# 生成结果
result = test_model(
    instruction=your_instruction,
    context=your_context,
    max_tokens=300
)

print("📤 模型输出:")
print("-" * 70)
print(result)
print("-" * 70)

# 尝试解析和美化JSON结果
try:
    import json
    json_result = json.loads(result)
    
    print("\n✅ JSON解析成功！")
    print("\n📊 结构化数据:")
    print(json.dumps(json_result, indent=2, ensure_ascii=False))
    
    # 提取关键信息
    print("\n🔍 关键信息提取:")
    for key, value in json_result.items():
        if not isinstance(value, (dict, list)):
            print(f"   • {key}: {value}")
        elif isinstance(value, list):
            print(f"   • {key}: [{len(value)} 项]")
    
except json.JSONDecodeError as e:
    print(f"\n⚠️  JSON解析失败")
    print(f"   错误: {str(e)[:100]}")
    print(f"\n💡 提示:")
    print(f"   • 模型可能需要更多训练来生成有效的JSON")
    print(f"   • 尝试更改指令或提供更详细的上下文")
    print(f"   • 增加BATCH_SIZE或训练轮数可能会改善结果")

print("\n" + "="*70)
print("💡 提示: 修改上面代码中的 your_instruction 和 your_context 来测试其他指令")

## 🎯 步骤9：交互式推理（自定义输入）

## 🧪 步骤8：测试模型效果

## 💾 步骤7：保存训练好的模型

## 🚀 步骤5：开始训练

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import load_dataset

print("🔧 Setting up training with CodeLlama...")

# ============================================================
# 配置参数 - CodeLlama (英语/荷兰语代码生成优化)
# ============================================================

MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
OUTPUT_DIR = "/content/drive/MyDrive/gis-models/codellama-gis-lora"  # 保存到Google Drive
TRAIN_FILE = "data/training/training_data_train.json"
VAL_FILE = "data/training/training_data_val.json"

# 训练参数（T4 GPU内存优化版 - 14GB显存限制）
NUM_EPOCHS = 3
BATCH_SIZE = 1  # Per-device batch size
GRADIENT_ACCUMULATION = 2  # Reduced from 4 → effective batch = 2
LEARNING_RATE = 2e-4  # CodeLlama推荐学习率
MAX_LENGTH = 512  # Reduced from 768 → significant memory savings

# LoRA参数（代码生成任务优化 - T4优化版）
LORA_R = 32  # Reduced from 64 → less trainable parameters
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

print(f"📦 Model: {MODEL_NAME}")
print(f"💾 Output: {OUTPUT_DIR}")
print(f"📊 Data: FILE-LEVEL (complete workflows)")
print(f"📈 Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, Accumulation: {GRADIENT_ACCUMULATION}, LR: {LEARNING_RATE}")
print(f"📄 Max Length: {MAX_LENGTH}")
print(f"🎯 LoRA Rank: {LORA_R} (T4-optimized)")
print(f"🎯 Optimized for: T4 GPU (14GB VRAM) - memory-efficient config")

In [ ]:
# 加载tokenizer (CodeLlama)
print("📖 Loading CodeLlama tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side="right"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer loaded: vocab_size={len(tokenizer)}")

In [ ]:
# ⚠️ 必须在model loading前运行！升级关键库
print("🔧 Upgrading transformers, accelerate, and installing bitsandbytes...")

# 升级transformers到兼容版本
!pip install -q --upgrade transformers==4.46.0

# 升级accelerate到最新稳定版修复optimizer.train()错误
!pip install -q --upgrade accelerate>=1.0.0

# 安装bitsandbytes用于8-bit优化器（节省显存）
!pip install -q bitsandbytes

print("✅ Libraries upgraded!")
print(f"  transformers: 4.46.0")
print(f"  accelerate: >=1.0.0 (latest stable, fixes optimizer.train() bug)")
print(f"  bitsandbytes: installed (8-bit optimizer for memory efficiency)")

In [ ]:
# 确保导入了必要的库
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import torch

# 使用float16加载模型（不量化，稳定适配T4显存）
print("🤖 Loading CodeLlama-7B with float16 and device offloading...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# 训练中必须禁用缓存以配合梯度检查点
model.config.use_cache = False

# 启用梯度检查点（节省显存）
model.gradient_checkpointing_enable()

print("✅ CodeLlama base model loaded (float16 with device offloading)")

# 应用LoRA
print("🔧 Applying LoRA...")
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied!")

In [ ]:
from datasets import load_dataset

# 准备数据集
print("📊 Preparing datasets...")

train_data = load_dataset('json', data_files=TRAIN_FILE, split='train')
eval_data = load_dataset('json', data_files=VAL_FILE, split='train')

print(f"  Train: {len(train_data)} samples")
print(f"  Val: {len(eval_data)} samples")

# 格式化prompt (CodeLlama优化格式)
def format_prompt(example):
    instruction = example['instruction']
    input_text = example.get('input', '')
    output = example['output']

    # CodeLlama更适合直接的代码生成格式
    if input_text:
        prompt = f"""You are a GIS workflow code generator. Generate complete JSON workflow code based on the instruction.

Instruction: {instruction}
Context: {input_text}

JSON Code:
{output}"""
    else:
        prompt = f"""You are a GIS workflow code generator. Generate complete JSON workflow code based on the instruction.

Instruction: {instruction}

JSON Code:
{output}"""

    return {"text": prompt}

train_data = train_data.map(format_prompt, remove_columns=train_data.column_names)
eval_data = eval_data.map(format_prompt, remove_columns=eval_data.column_names)

# Tokenize
def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("🔄 Tokenizing...")
train_dataset = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names,
    desc="Tokenizing train"
)

eval_dataset = eval_data.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_data.column_names,
    desc="Tokenizing val"
)

print("✅ Datasets ready!")

In [ ]:
# 配置训练
print("⚙️ Configuring training (T4-optimized)...")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,  # Reduced from 100
    logging_steps=10,
    save_steps=300,  # Reduced from 500
    eval_steps=300,  # Reduced from 500
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=True,  # Enable fp16 compute
    bf16=False,
    optim="adamw_8bit",  # 8-bit AdamW for memory efficiency (requires bitsandbytes)
    lr_scheduler_type="cosine",
    save_total_limit=2,  # Reduced from 3 to save disk space
    report_to="none",
    logging_dir=f"{OUTPUT_DIR}/logs",
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
    push_to_hub=False,
    gradient_checkpointing=True,  # Memory savings via checkpointing
    max_grad_norm=1.0,  # Gradient clipping
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,  # Use processing_class instead of tokenizer (new API)
    data_collator=data_collator,
)

print("✅ Trainer ready!")
print("\n" + "="*70)
print("🚀 Starting training (T4-optimized, may take 1-2 hours)...")
print("="*70)

# 训练前清理显存，减少碎片
import gc
gc.collect()
torch.cuda.empty_cache()

# 开始训练
trainer.train()

print("\n" + "="*70)
print("🎉 Training completed!")
print("="*70)

In [ ]:
# 保存模型
print("💾 Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

# 保存训练信息
import json
training_info = {
    "model_name": MODEL_NAME,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "train_samples": len(train_dataset),
    "val_samples": len(eval_dataset),
}

with open(f"{OUTPUT_DIR}/training_info.json", 'w') as f:
    json.dump(training_info, f, indent=2)

print("\n📊 Training Summary:")
for key, value in training_info.items():
    print(f"  {key}: {value}")

## 🧪 步骤6：测试模型

In [ ]:
# 快速测试 (CodeLlama)
print("🧪 Testing CodeLlama model inference...")

test_instruction = "Create a new MS cable object at coordinates (186355533, 439556907)"
test_context = "Application: PowerGrid | Database: ND | Steps: 5"

prompt = f"""You are a GIS workflow code generator. Generate complete JSON workflow code based on the instruction.

Instruction: {test_instruction}
Context: {test_context}

JSON Code:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("\n🔮 Generating...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = response.split("JSON Code:")[-1].strip()

print("\n" + "="*70)
print("📝 Test Result:")
print("="*70)
print(f"Instruction: {test_instruction}")
print(f"Context: {test_context}")
print(f"\nGenerated Output:\n{response[:500]}...")
print("="*70)

## 🧪 步骤6：在测试集上评估模型

In [ ]:
# 📊 在测试集上评估模型性能
import json
import numpy as np
from pathlib import Path

print("\n" + "="*70)
print("📊 模型性能评估 - 测试集")
print("="*70)

# ===================================================================
# 加载测试数据
# ===================================================================
print("\n📂 加载测试数据...")

# 使用验证集作为测试集
test_data = eval_data.to_list()
NUM_EVAL_SAMPLES = min(50, len(test_data))  # 评估前50个样本

print(f"✅ 加载了 {len(test_data)} 个验证样本")
print(f"📊 将评估前 {NUM_EVAL_SAMPLES} 个样本\n")

# ===================================================================
# 定义评估指标函数
# ===================================================================
def is_valid_json(text):
    """检查文本是否是有效的JSON"""
    try:
        json.loads(text)
        return True
    except (json.JSONDecodeError, TypeError):
        return False

def get_step_count(text):
    """从JSON中提取步骤数"""
    try:
        obj = json.loads(text)
        steps = obj.get("workflow", {}).get("steps", [])
        return len(steps)
    except:
        return 0

def extract_json_from_text(text):
    """从生成的文本中提取JSON部分"""
    # 尝试找到JSON的开始
    if "{" in text:
        start = text.index("{")
        # 尝试找到JSON的结束
        if "}" in text[start:]:
            end = text.rindex("}") + 1
            return text[start:end]
    return text

# ===================================================================
# 执行批量评估
# ===================================================================
print("🧪 开始评估...")
print("="*70)

evaluation_results = {
    "json_valid": [],
    "step_counts": [],
    "instruction_lengths": [],
    "generation_successful": [],
}

json_valid_count = 0
successful_count = 0

for i in range(NUM_EVAL_SAMPLES):
    sample = test_data[i]
    instruction = sample.get("instruction", "")
    context = sample.get("input", "")

    # 显示进度
    if (i + 1) % 10 == 0:
        print(f"进度: {i + 1}/{NUM_EVAL_SAMPLES}")

    try:
        # 构建prompt
        prompt = f"""You are a GIS workflow code generator. Generate complete JSON workflow code based on the instruction.

Instruction: {instruction}
Context: {context}

JSON Code:
"""

        # Tokenize
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        # 生成
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,  # 限制长度加快速度
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
            )

        # 解码
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # 提取JSON部分
        if "JSON Code:" in generated_text:
            generated_code = generated_text.split("JSON Code:")[-1].strip()
        else:
            generated_code = generated_text

        generated_code = extract_json_from_text(generated_code)

        # 评估结果
        is_valid = is_valid_json(generated_code)
        step_count = get_step_count(generated_code) if is_valid else 0

        evaluation_results["generation_successful"].append(True)
        evaluation_results["json_valid"].append(is_valid)
        evaluation_results["step_counts"].append(step_count)
        evaluation_results["instruction_lengths"].append(len(instruction.split()))

        if is_valid:
            json_valid_count += 1

        successful_count += 1

    except Exception as e:
        # 生成失败
        evaluation_results["generation_successful"].append(False)
        evaluation_results["json_valid"].append(False)
        evaluation_results["step_counts"].append(0)
        evaluation_results["instruction_lengths"].append(len(instruction.split()))
        print(f"  ⚠️ 样本 {i+1} 生成失败: {str(e)[:50]}")

# ===================================================================
# 计算并显示评估结果
# ===================================================================
print("\n" + "="*70)
print("📊 评估结果摘要")
print("="*70)

total_samples = len(evaluation_results["generation_successful"])

# 1. 生成成功率
print(f"\n✅ 生成成功率:")
print(f"   成功: {successful_count}/{total_samples} ({successful_count/total_samples:.1%})")

# 2. JSON有效性
if successful_count > 0:
    print(f"\n✅ JSON有效性 (成功生成的代码中):")
    print(f"   有效: {json_valid_count}/{successful_count} ({json_valid_count/successful_count:.1%})")
else:
    print(f"\n❌ 所有生成均失败")

# 3. 步骤数统计
valid_step_counts = [s for s, v in zip(evaluation_results["step_counts"], evaluation_results["json_valid"]) if v]
if valid_step_counts:
    print(f"\n📍 有效代码的步骤数统计:")
    print(f"   平均: {np.mean(valid_step_counts):.1f}")
    print(f"   中位: {np.median(valid_step_counts):.1f}")
    print(f"   范围: {int(np.min(valid_step_counts))}-{int(np.max(valid_step_counts))}")

# 4. 指令长度分析
print(f"\n📝 指令长度 (词数):")
print(f"   平均: {np.mean(evaluation_results['instruction_lengths']):.1f}")
print(f"   最长: {int(np.max(evaluation_results['instruction_lengths']))}")

# 5. 总体质量评估
if successful_count > 0:
    quality_rate = json_valid_count / successful_count
else:
    quality_rate = 0

quality_stars = (
    "⭐⭐⭐⭐⭐" if quality_rate > 0.9 else
    "⭐⭐⭐⭐" if quality_rate > 0.7 else
    "⭐⭐⭐" if quality_rate > 0.5 else
    "⭐⭐" if quality_rate > 0.3 else
    "⭐"
)

print(f"\n🎯 总体质量评估:")
print(f"   {quality_stars} ({quality_rate:.1%} 有效率)")

# ===================================================================
# 改进建议
# ===================================================================
print("\n" + "="*70)
print("💡 改进建议")
print("="*70)

if quality_rate < 0.5:
    print("""
⚠️  模型质量较低，建议:
   1. 增加训练轮数 (num_epochs: 3 → 5)
   2. 使用更多训练数据
   3. 降低学习率 (lr: 2e-4 → 1e-4)
   4. 增加预热步骤 (warmup_steps: 50 → 200)
   5. 尝试不同的LoRA秩 (r: 32 → 64)
    """)
elif quality_rate < 0.7:
    print("""
✓ 模型质量一般，可以继续改进:
   1. 增加训练轮数
   2. 微调学习率
   3. 收集更多多样化的训练样本
    """)
else:
    print("""
✅ 模型质量良好！
   • 可以开始在真实场景中使用
   • 继续收集失败样例以进一步改进
   • 考虑在其他GIS工作流上进行微调
    """)

print("\n" + "="*70)
print("🎉 评估完成！")
print("="*70)